# Part 2: GitHub Contributor Analytics Pipeline

## Overview

| Property  | Value  |
|-----------|--------|
| Tasks     | 3      |
| Languages | Python |

### Objective
Build an ingestion and transformation pipeline using GitHub's REST API to analyze repository contributors.

### Data Source
- **Target Repository:** `apache/airflow`
- **Base URL:** `https://api.github.com`

### Endpoints
- `/repos/apache/airflow/commits`
- `/repos/apache/airflow/pulls`
- `/repos/apache/airflow/pulls/comments`
- `/repos/apache/airflow/issues`
- `/repos/apache/airflow/pulls/{pull_number}/reviews`

---
## Setup

Load environment variables and establish Snowflake connection.

In [1]:
import os
from dotenv import load_dotenv
import snowflake.connector

load_dotenv()

assert os.environ.get('GITHUB_TOKEN'), 'GITHUB_TOKEN not set. Check .env file.'
print('Environment loaded successfully.')

# Snowflake connection for data storage
from src.docs_pipeline.config import get_snowflake_params
conn = snowflake.connector.connect(**get_snowflake_params())
print('Snowflake connected.')

Environment loaded successfully.
Snowflake connected.


---
## Create Snowflake Tables

Drop existing tables (clean slate) and recreate them.

In [2]:
from src.github_pipeline.ddl import get_ddl, apply_ddl, drop_tables

# Drop existing tables for a clean slate
drop_tables(conn)

# Preview DDL
print(get_ddl())

# Recreate tables
apply_ddl(conn)

  Dropped CANDIDATE_NS_GH_RAW_COMMITS
  Dropped CANDIDATE_NS_GH_RAW_PULLS
  Dropped CANDIDATE_NS_GH_RAW_PR_COMMENTS
  Dropped CANDIDATE_NS_GH_RAW_ISSUES
  Dropped CANDIDATE_NS_GH_RAW_REVIEWS
All GitHub raw tables dropped.

-- GitHub Raw Data Tables

CREATE TABLE IF NOT EXISTS CANDIDATE_NS_GH_RAW_COMMITS (
    SHA             VARCHAR(40)     NOT NULL PRIMARY KEY,
    AUTHOR_NAME     VARCHAR(500),
    AUTHOR_LOGIN    VARCHAR(200),
    COMMITTED_DATE  TIMESTAMP_NTZ,
    MESSAGE         VARCHAR(5000),
    RAW_JSON        VARIANT         NOT NULL,
    LOADED_AT       TIMESTAMP_NTZ   DEFAULT CURRENT_TIMESTAMP()
);

CREATE TABLE IF NOT EXISTS CANDIDATE_NS_GH_RAW_PULLS (
    NUMBER          INTEGER         NOT NULL PRIMARY KEY,
    AUTHOR_LOGIN    VARCHAR(200),
    STATE           VARCHAR(20),
    TITLE           VARCHAR(2000),
    UPDATED_AT      TIMESTAMP_NTZ,
    CREATED_AT      TIMESTAMP_NTZ,
    RAW_JSON        VARIANT         NOT NULL,
    LOADED_AT       TIMESTAMP_NTZ   DEFAULT CURRENT_

---
## Task 1: Data Ingestion

### Requirements
- Ingest from all endpoints listed above
- Store each endpoint as a separate dataset
- Output: Row count per endpoint after ingestion completes

Data is stored in Snowflake and supports resume. If interrupted, re-running
this cell will continue from the last checkpoint.

In [3]:
from src.github_pipeline.ingestion import run_ingestion

# Pass conn for Snowflake storage + resume; omit for fetch-only mode
data = run_ingestion(conn=conn, from_snowflake=True)

print('\n--- Fetched Row Counts ---')
for key, df in data.items():
    print(f'  {key}: {len(df)} rows')

Loading data from Snowflake raw tables (skipping API)...
  Reading CANDIDATE_NS_GH_RAW_COMMITS from Snowflake...
  Loaded 35795 rows from CANDIDATE_NS_GH_RAW_COMMITS
  Reading CANDIDATE_NS_GH_RAW_PULLS from Snowflake...
  Loaded 42495 rows from CANDIDATE_NS_GH_RAW_PULLS
  Reading CANDIDATE_NS_GH_RAW_PR_COMMENTS from Snowflake...
  Loaded 103495 rows from CANDIDATE_NS_GH_RAW_PR_COMMENTS
  Reading CANDIDATE_NS_GH_RAW_ISSUES from Snowflake...
  Loaded 0 rows from CANDIDATE_NS_GH_RAW_ISSUES
  Reading CANDIDATE_NS_GH_RAW_REVIEWS from Snowflake...
  Loaded 1975 rows from CANDIDATE_NS_GH_RAW_REVIEWS

--- Fetched Row Counts ---
  commits: 35795 rows
  pulls: 42495 rows
  pr_comments: 103495 rows
  issues: 0 rows
  reviews: 1975 rows


---
## Task 2: Transformation

### Requirements
Build a contributor analytics dataset by joining ingested data.

### Output Schema

| Column       | Description                            |
|-------------|----------------------------------------|
| author      | GitHub username (login)                 |
| commits     | Commit count                            |
| prs         | Pull request count (as author)          |
| comments    | PR comment count                        |
| reviews     | Review count                            |
| score       | Weighted score (formula below), max 100 |
| tier        | Classification based on activity        |
| overall_rank| Global rank by score (1 = highest)      |
| tier_rank   | Rank within tier                        |
| percentile  | Score percentile (0-100 scale)          |

### Scoring Formula
```
raw_score = (commits x 5) + (prs x 10) + (comments x 2) + (reviews x 3)
score = min(raw_score, 100)
```

### Tier Definitions

| Tier        | Criteria              |
|------------|----------------------|
| core       | commits + prs >= 20   |
| active     | commits + prs >= 5    |
| contributor| commits + prs >= 1    |
| observer   | commits + prs = 0     |

In [4]:
from src.github_pipeline.transformation import process_contributors, print_summary

contributors_df = process_contributors(data)
print(f'Contributors DataFrame: {contributors_df.shape[0]} rows, {contributors_df.shape[1]} columns')
contributors_df.head(10)

Contributors DataFrame: 8536 rows, 11 columns


,author,commits,prs,comments,reviews,raw_score,score,tier,overall_rank,tier_rank,percentile
0,potiuk,0,5249,10408,352,74362,100,core,1,1,100.00
1,kaxil,3,2044,5540,12,31571,100,core,2,2,99.99
2,ashb,0,1020,8045,5,26305,100,core,3,3,99.98
3,Jarek Potiuk,4505,0,0,0,22525,100,core,4,4,99.96
4,uranusjr,0,712,6484,110,20418,100,core,5,5,99.95
5,jedcunningham,0,853,4296,66,17320,100,core,6,6,99.94
6,mik-laj,0,930,3523,1,16349,100,core,7,7,99.93
7,amoghrajesh,0,881,3269,24,15420,100,core,8,8,99.92
8,dstandish,38,786,2942,41,14057,100,core,9,9,99.91
9,eladkal,275,691,2572,159,13906,100,core,10,10,99.89


### Required Output

In [5]:
print_summary(contributors_df)

Top 10 Contributors by Score:
          author  score  tier
0         potiuk    100  core
1          kaxil    100  core
2           ashb    100  core
3   Jarek Potiuk    100  core
4       uranusjr    100  core
5  jedcunningham    100  core
6        mik-laj    100  core
7    amoghrajesh    100  core
8      dstandish    100  core
9        eladkal    100  core

Tier Distribution:
tier
contributor    7067
active          894
core            351
observer        224
Name: count, dtype: int64

Summary:
Total contributors: 8536
Min score: 2
Max score: 100
Count achieving max score (100): 574


---
## Task 3: Export to Google Sheets

### Requirements
- Export contributor analytics to Google Sheets
- Uses `GITHUB_SHEETS_SPREADSHEET_ID` and `GOOGLE_APPLICATION_CREDENTIALS` from `.env`

In [6]:
from src.github_pipeline.export_sheets import export_to_sheets
from src.github_pipeline.config import get_sheets_spreadsheet_id

spreadsheet_id = get_sheets_spreadsheet_id()
result = export_to_sheets(contributors_df, spreadsheet_id=spreadsheet_id)

print(f"\nExport complete!")
print(f"View your sheet at: https://docs.google.com/spreadsheets/d/{result['spreadsheet_id']}/edit")

Using Google Service Account: sheets-snowflae-access@firegram-1r.iam.gserviceaccount.com
Wrote 8537 rows to 'Part2_Contributors'

Export complete!
View your sheet at: https://docs.google.com/spreadsheets/d/1Wcy2UrqUk-MzCZpsgPWq9QWzrLxZgxTTADj1z-dItjw/edit


---
## Cleanup

In [ ]:
conn.close()
print('Snowflake connection closed.')